## Introduction

Ce Notebook presente le code pour récuperer les données nécessaires pour analyser les emissions dans la région l'île de France. Pour étudier notre problématique rigoureusement, on doit utiliser des données composant:

1) On a besoin des concentrations de particules qui puissent nuire à la population. On utilise le NO2, PM2,5 et le O3. On note que la récupération de ses données n'est pas si directe comme on pourrait l'imaginer. En effet, le site the Airparif (qui est le site donnant les données sur la qualité de l'air de toute la région de l'île de France) ne donne que les valeurs des stations qui mesurent la qualité de l'air. Or, on a besoin de pouvoir une vision spatiale de toutes les communes et pas que de certaines stations. Alors, la récupération de ces données devient complexe. Néanmoins, Airparifs offre aussi des cartes intéractives qui donnent les niveaux de concentrations des particules. Donc, en utilisant ces cartes avec les données déjà modélisées sur toute l'Île de France, on pourrait réaliser notre étude. J'ai inspectionné le Slider donnant les cartes et j'ai trouvé d'oú je pouvais extraire les données. 

2) Pour pouvoir utiliser les données on devra regrouper les données par communes de l'INSEE, donc on a besoin des coordonnées de centroïde de chaque commune. Pour cela on utilise la geo API de datagouv.fr, qui donne des coordonnées des communes.

L'idée est d'utiliser les coordonées des centroïdes des communes et de "scraper" la concentration des particules nuisant la qualité de l'air directement des données qui sont presentées sur la carte (qui modélisent directement la concentration des particules aux alentours des stations de mesure)


In [32]:
import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
import xml 
import geopandas as gp
import requests
from functions_emissions import extract_values 

In [33]:
# Données du Geoserver de Airparif 

url_base = "https://namek.airparif.fr/geoserver/Moyenne_annuelle/ows?service=WMS&version=1.3.0&request=GetFeatureInfo"
years = range(2007,2008)
particles = {"no2"}


In [37]:
# Données des centroïdes des communes de l'île de france

url_communes = "https://geo.api.gouv.fr/communes"
departement_idf = ["75","77","78","91","92","93","94","95"]

data_centroides = []

for dep in departement_idf:
    parametres = {"codeDepartement": dep, "type": "commune-actuelle,arrondissement-municipal","fields": "nom,code,codeDepartement,centre", "format": "json"}
    r = requests.get(url_communes, params = parametres).json()
    data_centroides += r
    
ile_de_fr_commune = pd.DataFrame({
    "INSEE_Commune" : [com["code"] for com in data_centroides],
    "INSEE_Departement" : [com["codeDepartement"] for com in data_centroides],
    "Nom_Commune" : [com["nom"] for com in data_centroides],
    "Latitude" : [com["centre"]["coordinates"][1] for com in data_centroides],
    "Longitude" : [com["centre"]["coordinates"][0] for com in data_centroides]})

ile_de_fr_commune.drop(index=0).reset_index(drop=True) # la premiere ligne est Paris en general, donc on l'enleve

,INSEE_Commune,INSEE_Departement,Nom_Commune,Latitude,Longitude
0,75101,75,Paris 1er Arrondissement,48.8620,2.3359
1,75102,75,Paris 2e Arrondissement,48.8677,2.3411
2,75103,75,Paris 3e Arrondissement,48.8626,2.3593
3,75104,75,Paris 4e Arrondissement,48.8541,2.3569
4,75105,75,Paris 5e Arrondissement,48.8454,2.3514
...,...,...,...,...,...
1280,95676,95,Villers-en-Arthies,49.0864,1.7322
1281,95678,95,Villiers-Adam,49.0704,2.2383
1282,95680,95,Villiers-le-Bel,49.0080,2.4036
1283,95682,95,Villiers-le-Sec,49.0751,2.3862


In [38]:
# Scraper les données de concentration pour chaque commune dans l'île-de-France
data_concentration = []

for communes in range(ile_de_fr_commune.shape[0]):
    for particule in particles:
        for year in years:
            concentration = extract_values(url_base, ile_de_fr_commune["Longitude"].loc[communes], ile_de_fr_commune["Latitude"].loc[communes], particule, year)
            dict_year_particle = {"Year": year, "Particle": particule, "Commune": ile_de_fr_commune["Nom_Commune"].loc[communes], "Code_INSEE": ile_de_fr_commune["INSEE_Commune"].loc[communes], "Concentration": concentration}
            data_concentration.append(dict_year_particle)


In [ ]:
# Construction du Panel (Commune, année, Partibule_polluante) pour l'analyse




[{'Year': 2007,
  'Particle': 'no2',
  'Commune': 'Paris',
  'Code_INSEE': '75056',
  'Concentration': 59.6128117799759},
 {'Year': 2007,
  'Particle': 'no2',
  'Commune': 'Paris 1er Arrondissement',
  'Code_INSEE': '75101',
  'Concentration': 42.57449026107787},
 {'Year': 2007,
  'Particle': 'no2',
  'Commune': 'Paris 2e Arrondissement',
  'Code_INSEE': '75102',
  'Concentration': 41.47160574123265},
 {'Year': 2007,
  'Particle': 'no2',
  'Commune': 'Paris 3e Arrondissement',
  'Code_INSEE': '75103',
  'Concentration': 44.2056036233902},
 {'Year': 2007,
  'Particle': 'no2',
  'Commune': 'Paris 4e Arrondissement',
  'Code_INSEE': '75104',
  'Concentration': 46.53999801874161},
 {'Year': 2007,
  'Particle': 'no2',
  'Commune': 'Paris 5e Arrondissement',
  'Code_INSEE': '75105',
  'Concentration': 51.34873713135718},
 {'Year': 2007,
  'Particle': 'no2',
  'Commune': 'Paris 6e Arrondissement',
  'Code_INSEE': '75106',
  'Concentration': 36.94101240038873},
 {'Year': 2007,
  'Particle': 'n